# Parte 1 · Pesos de OLS, ridge, k-NN y árbol (1.1, 1.2 y 1.3)

**Autora:** Sofía Isabela Sandoval Vásquez · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por la autora.

Datos: California Housing, estandarizados como en 0.1. Para cada modelo se obtiene la matriz $W$ (consultas × entrenamiento) con `src/pesos.py` y se comprueba que $W y$ reproduce el `.predict()` en las primeras 20 filas de prueba.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

from datos import cargar_california
from pesos import pesos_ols, pesos_ridge, pesos_ridge_penalizando_unos, pesos_knn, pesos_arbol

X_train, X_test, y_train, y_test, _ = cargar_california()
q = X_test.iloc[:20]
n = len(X_train)


def resumen(w, pred):
    return {"error máx. |Wy − predict|": np.abs(w @ y_train - pred).max(),
            "Σw mínima": w.sum(1).min(), "Σw máxima": w.sum(1).max(),
            "pesos ≠ 0 (prom.)": (w != 0).sum(1).mean(), "pesos < 0 (prom.)": (w < 0).sum(1).mean()}

## 1.1 · OLS

> **Pregunta 1.1.** Muestren que $w(x)^\top = \tilde x^\top (\tilde X^\top \tilde X)^{-1} \tilde X^\top$ y que $\sum_i w_i(x) = 1$ para cualquier $x$.

Con $\tilde X$ la matriz de diseño con la columna de unos, OLS estima $\hat\beta = (\tilde X^\top \tilde X)^{-1} \tilde X^\top y$, así que

$$\hat y(x) = \tilde x^\top \hat\beta = \underbrace{\tilde x^\top (\tilde X^\top \tilde X)^{-1} \tilde X^\top}_{w(x)^\top}\, y .$$

Para la suma, el vector de unos es la primera columna de $\tilde X$: $\mathbf 1 = \tilde X e_1$. Entonces

$$\sum_i w_i(x) = w(x)^\top \mathbf 1 = \tilde x^\top (\tilde X^\top \tilde X)^{-1} \tilde X^\top \tilde X e_1 = \tilde x^\top e_1 = 1 .$$

Es lo mismo que decir que $H\mathbf 1 = \mathbf 1$: la matriz sombrero proyecta sobre el espacio columna de $\tilde X$, y el vector de unos ya está ahí, así que la proyección no lo cambia.

In [2]:
ols = LinearRegression().fit(X_train, y_train)
W_ols = pesos_ols(X_train, q)
assert np.abs(W_ols @ y_train - ols.predict(q)).max() < 1e-8
pd.Series(resumen(W_ols, ols.predict(q)))

error máx. |Wy − predict|    6.661338e-15
Σw mínima                    1.000000e+00
Σw máxima                    1.000000e+00
pesos ≠ 0 (prom.)            1.651200e+04
pesos < 0 (prom.)            5.163550e+03
dtype: float64

## 1.2 · Ridge con λ = 100

> **Pregunta 1.2 (a).** Muestren que $w(x)^\top = \frac{1}{n}\mathbf 1^\top + (x - \bar x)^\top (X_c^\top X_c + \lambda I)^{-1} X_c^\top$.

scikit-learn minimiza $\lVert y - \beta_0 \mathbf 1 - X\beta \rVert^2 + \lambda \lVert \beta \rVert^2$, sin penalizar $\beta_0$. Derivando respecto a $\beta_0$ e igualando a cero: $\hat\beta_0 = \bar y - \bar x^\top \beta$. Al sustituir, el problema queda $\lVert (y - \bar y \mathbf 1) - X_c \beta \rVert^2 + \lambda \lVert \beta \rVert^2$, cuya solución es

$$\hat\beta = (X_c^\top X_c + \lambda I)^{-1} X_c^\top (y - \bar y \mathbf 1) = (X_c^\top X_c + \lambda I)^{-1} X_c^\top y ,$$

porque las columnas de $X_c$ están centradas ($X_c^\top \mathbf 1 = 0$). La predicción es

$$\hat y(x) = \hat\beta_0 + x^\top \hat\beta = \bar y + (x - \bar x)^\top \hat\beta = \Big[\tfrac{1}{n}\mathbf 1^\top + (x - \bar x)^\top (X_c^\top X_c + \lambda I)^{-1} X_c^\top\Big]\, y .$$

Los pesos suman 1: el primer término aporta $\frac{1}{n}\mathbf 1^\top \mathbf 1 = 1$ y el segundo aporta 0, otra vez porque $X_c^\top \mathbf 1 = 0$.

In [3]:
ridge = Ridge(alpha=100).fit(X_train, y_train)
W_ridge = pesos_ridge(X_train, q, 100)
assert np.abs(W_ridge @ y_train - ridge.predict(q)).max() < 1e-8
pd.Series(resumen(W_ridge, ridge.predict(q)))

error máx. |Wy − predict|    8.881784e-16
Σw mínima                    1.000000e+00
Σw máxima                    1.000000e+00
pesos ≠ 0 (prom.)            1.651200e+04
pesos < 0 (prom.)            5.106800e+03
dtype: float64

Ahora la versión "obvia": se agrega una columna de unos y se penaliza junto con las demás, $w(x)^\top = \tilde x^\top (\tilde X^\top \tilde X + \lambda I)^{-1} \tilde X^\top$.

In [4]:
W_obvia = pesos_ridge_penalizando_unos(X_train, q, 100)

# esos pesos sí son los de un Ridge sin intercepto ajustado sobre [1, X]
unos = lambda X: np.column_stack([np.ones(len(X)), X])
ridge_sin_int = Ridge(alpha=100, fit_intercept=False).fit(unos(X_train), y_train)
assert np.abs(W_obvia @ y_train - ridge_sin_int.predict(unos(q))).max() < 1e-8

pd.DataFrame({
    "sklearn (intercepto libre)": resumen(W_ridge, ridge.predict(q)),
    "versión obvia vs. ese mismo .predict()": resumen(W_obvia, ridge.predict(q)),
}).T

,error máx. |Wy − predict|,Σw mínima,Σw máxima,pesos ≠ 0 (prom.),pesos < 0 (prom.)
sklearn (intercepto libre),8.881784e-16,1.00000,1.00000,16512.0,5106.8
versión obvia vs. ese mismo .predict(),1.247591e-02,0.99398,0.99398,16512.0,5124.9


In [5]:
diferencia = ridge.predict(q) - W_obvia @ y_train
pd.Series({
    "n / (n + λ)": n / (n + 100),
    "Σw de la versión obvia (todas las consultas)": W_obvia.sum(1).mean(),
    "λ/(n + λ) · ȳ": 100 / (n + 100) * y_train.mean(),
    "diferencia de predicción (mín.)": diferencia.min(),
    "diferencia de predicción (máx.)": diferencia.max(),
})

n / (n + λ)                                     0.993980
Σw de la versión obvia (todas las consultas)    0.993980
λ/(n + λ) · ȳ                                   0.012476
diferencia de predicción (mín.)                 0.012476
diferencia de predicción (máx.)                 0.012476
dtype: float64

> **Pregunta 1.2 (b).** ¿Cuadra con `.predict()`? ¿Cuánto suman ahora los pesos? Expliquen.

No cuadra: la versión obvia se aleja del `.predict()` de scikit-learn en unas 0.0125 unidades (unos $1 250) en todas las consultas, y sus pesos ya no suman 1 sino 0.99398 para cualquier $x$. La razón es que ahora el intercepto también se encoge hacia 0, y el argumento de 1.1 deja de valer: $(\tilde X^\top \tilde X + \lambda I)^{-1} \tilde X^\top \tilde X \neq I$.

Como las $X$ están estandarizadas con el entrenamiento, cada columna tiene media 0 y $\tilde X^\top \mathbf 1 = (n, 0, \dots, 0)^\top$. Además $\tilde X^\top \tilde X$ es diagonal por bloques (el bloque del intercepto vale $n$), así que

$$\sum_i w_i(x) = \tilde x^\top (\tilde X^\top \tilde X + \lambda I)^{-1} (n, 0, \dots, 0)^\top = \frac{n}{n + \lambda} = \frac{16\,512}{16\,612} \approx 0.99398 .$$

Las pendientes no cambian; solo el intercepto pasa de $\bar y$ a $\frac{n}{n + \lambda}\bar y$, y por eso la diferencia de predicción es la misma constante $\frac{\lambda}{n + \lambda}\bar y$ en todas las consultas, como muestra la tabla. En la práctica, penalizar el intercepto hace que el modelo dependa de dónde está el origen de $y$, que es justo lo que scikit-learn evita al dejarlo libre.

## 1.3 · k-NN (k = 30) y árbol (profundidad 6)

> **Pregunta 1.3.** Escriban $w(x)$ para `KNeighborsRegressor(n_neighbors=30)` y para `DecisionTreeRegressor(max_depth=6)`.

**k-NN.** La predicción es el promedio de las $y$ de los $k$ vecinos más cercanos $N_k(x)$:

$$w_i(x) = \frac{\mathbf 1\{x_i \in N_k(x)\}}{k} .$$

Cada vecino pesa $1/30$ y el resto pesa 0. Los vecinos se obtienen con `.kneighbors()`.

**Árbol.** La predicción es el promedio de las $y$ de entrenamiento que caen en la misma hoja $L(x)$ que $x$:

$$w_i(x) = \frac{\mathbf 1\{x_i \in L(x)\}}{|L(x)|} .$$

La hoja de cada fila se obtiene con `.apply()`. En los dos casos los pesos son no negativos y suman 1, y el número de pesos distintos de cero es $k$ (k-NN) o el tamaño de la hoja (árbol).

In [6]:
knn = KNeighborsRegressor(n_neighbors=30).fit(X_train, y_train)
arbol = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X_train, y_train)
W_knn = pesos_knn(knn, q, n)
W_arbol = pesos_arbol(arbol, X_train, q)
assert np.abs(W_knn @ y_train - knn.predict(q)).max() < 1e-8
assert np.abs(W_arbol @ y_train - arbol.predict(q)).max() < 1e-8

### Resumen de los cuatro modelos (primeras 20 filas de prueba)

In [7]:
pd.DataFrame({
    "OLS": resumen(W_ols, ols.predict(q)),
    "Ridge (λ = 100)": resumen(W_ridge, ridge.predict(q)),
    "k-NN (k = 30)": resumen(W_knn, knn.predict(q)),
    "Árbol (prof. 6)": resumen(W_arbol, arbol.predict(q)),
}).T

,error máx. |Wy − predict|,Σw mínima,Σw máxima,pesos ≠ 0 (prom.),pesos < 0 (prom.)
OLS,6.661338e-15,1.0,1.0,16512.0,5163.55
Ridge (λ = 100),8.881784e-16,1.0,1.0,16512.0,5106.80
k-NN (k = 30),8.881784e-16,1.0,1.0,30.0,0.00
Árbol (prof. 6),3.996803e-15,1.0,1.0,535.3,0.00


Los cuatro reproducen el `.predict()` con un error del orden de $10^{-15}$ y sus pesos suman 1. La diferencia está en cuántas filas votan y cómo: OLS y ridge reparten el voto entre las 16 512 filas de entrenamiento, con miles de pesos negativos; k-NN usa exactamente 30 filas con el mismo peso, y el árbol usa todas las filas de la hoja.